# 4.SQL and Dataframes

References:

* Spark-SQL, <https://spark.apache.org/docs/latest/sql-programming-guide.html#datasets-and-dataframes>


# 4.1  Example Walkthrough
Follow the Spark SQL and DataFrames examples below.

### Initialize PySpark

In [1]:
!pip install -q pyspark

In [2]:
# Show available CPU cores (useful for configuring Spark parallelism)
import multiprocessing
print(f"Available CPU cores: {multiprocessing.cpu_count()}")

Available CPU cores: 2


In [13]:
# Initialize PySpark
import os, sys
APP_NAME = "PySpark Lecture"
SPARK_MASTER="local[2]"
import pyspark
import pyspark.sql
from pyspark.sql import Row
conf=pyspark.SparkConf()
conf=pyspark.SparkConf().setAppName(APP_NAME).set("spark.local.dir", os.path.join(os.getcwd(), "tmp"))
sc = pyspark.SparkContext(master=SPARK_MASTER, conf=conf)
spark = pyspark.sql.SparkSession(sc).builder.appName(APP_NAME).getOrCreate()

print("PySpark initiated...")

ValueError: Cannot run multiple SparkContexts at once; existing SparkContext(app=PySpark Lecture, master=local[2]) created by __init__ at /tmp/ipykernel_6305/1993162062.py:10 

### Hello, World!

Loading data, mapping it and collecting the records into RAM...

In [10]:
!wget https://raw.githubusercontent.com/scalable-infrastructure/exercise-2026/main/data/example.csv

--2026-03-29 19:24:15--  https://raw.githubusercontent.com/scalable-infrastructure/exercise-2026/main/data/example.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 189 [text/plain]
Saving to: ‘example.csv.1’

example.csv.1       100%[===================>]     189  --.-KB/s    in 0s      

2026-03-29 19:24:15 (2.54 MB/s) - ‘example.csv.1’ saved [189/189]



In [5]:
# Load the text file using the SparkContext
csv_lines = sc.textFile("example.csv")

# Map the data to split the lines into a list
data = csv_lines.map(lambda line: line.split(","))

# Collect the dataset into local RAM
data.collect()

[['Russell Jurney', 'Relato', 'CEO'],
 ['Florian Liebert', 'Mesosphere', 'CEO'],
 ['Don Brown', 'Rocana', 'CIO'],
 ['Steve Jobs', 'Apple', 'CEO'],
 ['Donald Trump', 'The Trump Organization', 'CEO'],
 ['Russell Jurney', 'Data Syndrome', 'Principal Consultant']]

### Creating Rows

Creating `pyspark.sql.Rows` out of your data so you can create DataFrames...

In [6]:
# Convert the CSV into a pyspark.sql.Row
def csv_to_row(line):
    parts = line.split(",")
    row = Row(
      name=parts[0],
      company=parts[1],
      title=parts[2]
    )
    return row

# Apply the function to get rows in an RDD
rows = csv_lines.map(csv_to_row)

### Creating DataFrames from RDDs

Using the `RDD.toDF()` method to create a dataframe, registering the `DataFrame` as a temporary table with Spark SQL, and counting the jobs per person using Spark SQL.

In [12]:
# Convert to a pyspark.sql.DataFrame
rows_df = rows.toDF()

# Register the DataFrame for Spark SQL
rows_df.registerTempTable("executives")

# Generate a new DataFrame with SQL using the SparkSession
job_counts = spark.sql("""
SELECT
  name,
  COUNT(*) AS total
  FROM executives
  GROUP BY name
""")
job_counts.show()

# Go back to an RDD
job_counts.rdd.collect()

Py4JJavaError: An error occurred while calling o81.partitions.
: org.apache.hadoop.mapred.InvalidInputException: Input path does not exist: file:/content/NASA_access_log_Jul95
	at org.apache.hadoop.mapred.FileInputFormat.singleThreadedListStatus(FileInputFormat.java:306)
	at org.apache.hadoop.mapred.FileInputFormat.listStatus(FileInputFormat.java:245)
	at org.apache.hadoop.mapred.FileInputFormat.getSplits(FileInputFormat.java:334)
	at org.apache.spark.rdd.HadoopRDD.getPartitions(HadoopRDD.scala:233)
	at org.apache.spark.rdd.RDD.$anonfun$partitions$2(RDD.scala:301)
	at scala.Option.getOrElse(Option.scala:201)
	at org.apache.spark.rdd.RDD.partitions(RDD.scala:297)
	at org.apache.spark.rdd.MapPartitionsRDD.getPartitions(MapPartitionsRDD.scala:49)
	at org.apache.spark.rdd.RDD.$anonfun$partitions$2(RDD.scala:301)
	at scala.Option.getOrElse(Option.scala:201)
	at org.apache.spark.rdd.RDD.partitions(RDD.scala:297)
	at org.apache.spark.api.java.JavaRDDLike.partitions(JavaRDDLike.scala:62)
	at o

# 4.2–4.7 NASA Dataset

## 4.2
Create a Spark SQL table with fields for IP/Host and Response Code from the NASA log file.

In [11]:
# Exercise 4.2: Create a Spark SQL table with IP/Host and Response Code
print("Exercise 4.2: Create a Spark SQL table with IP/Host and Response Code")
# Download the NASA log file
!wget -q https://raw.githubusercontent.com/scalable-infrastructure/exercise-2026/main/data/NASA_access_log_Jul95

# Load the NASA log file
log_lines = sc.textFile("NASA_access_log_Jul95")
# Clean invalid lines same as task 3
valid_lines = log_lines.filter(lambda line: len(line.split()) > 8)
# Extract only the fields we need
# [0]=host [8]=response code
parsed = valid_lines.map(lambda line: (
    line.split()[0],   # host = first column (IP address)
    line.split()[8]    # response_code = 9th column
))
# Convert each tuple into a Spark Row
rows = parsed.map(lambda x: Row(host=x[0], response_code=x[1]))
# Create a DataFrame from the rows
df = spark.createDataFrame(rows)
# Register as a SQL table so we can query it
df.createOrReplaceTempView("nasa_logs")
print("Check table")
df.show(5)

Exercise 4.2: Create a Spark SQL table with IP/Host and Response Code


Py4JJavaError: An error occurred while calling o81.partitions.
: org.apache.hadoop.mapred.InvalidInputException: Input path does not exist: file:/content/NASA_access_log_Jul95
	at org.apache.hadoop.mapred.FileInputFormat.singleThreadedListStatus(FileInputFormat.java:306)
	at org.apache.hadoop.mapred.FileInputFormat.listStatus(FileInputFormat.java:245)
	at org.apache.hadoop.mapred.FileInputFormat.getSplits(FileInputFormat.java:334)
	at org.apache.spark.rdd.HadoopRDD.getPartitions(HadoopRDD.scala:233)
	at org.apache.spark.rdd.RDD.$anonfun$partitions$2(RDD.scala:301)
	at scala.Option.getOrElse(Option.scala:201)
	at org.apache.spark.rdd.RDD.partitions(RDD.scala:297)
	at org.apache.spark.rdd.MapPartitionsRDD.getPartitions(MapPartitionsRDD.scala:49)
	at org.apache.spark.rdd.RDD.$anonfun$partitions$2(RDD.scala:301)
	at scala.Option.getOrElse(Option.scala:201)
	at org.apache.spark.rdd.RDD.partitions(RDD.scala:297)
	at org.apache.spark.api.java.JavaRDDLike.partitions(JavaRDDLike.scala:62)
	at o

## 4.3
Run an SQL query that outputs the number of occurrences of each HTTP response code.

## 4.4
Implement the same query using the DataFrame API.

## 4.5
Cache the DataFrame and run the same query again. Measure the runtime for caching and for executing the query.

## 4.6 Performance Analysis — Weak Scaling
* Create RDDs with 2x, 4x, 8x, and 16x the size of the NASA log dataset. Persist the dataset in the Spark cache. Use an appropriate number of cores (e.g. 8 or 16).
* Measure and plot the response times for all datasets using a constant number of cores.
* Plot the results.
* Explain the results.


## 4.7 Performance Analysis — Strong Scaling

* **Measure the runtime for the query for 1, 2, 4 worker threads (`local[n]`) for 1x and 16x datasets.** Datasets should be cached in memory. Note that the Colab environment only has two cores.
* Compute the speedup and efficiency.
* Plot the results.
* Explain the results.

In [ ]:
# Exercise 4.2: Create a Spark SQL table with IP/Host and Response Code
print("Exercise 4.2: Create a Spark SQL table with IP/Host and Response Code")
